In [0]:
dbutils.widgets.text("table_name","")

In [0]:
table_name = dbutils.widgets.get('table_name')


In [0]:
dbutils.widgets.text("catalog_name","")

In [0]:
catalog_name = dbutils.widgets.get('catalog_name')

In [0]:
print(catalog_name)
print(table_name)

In [0]:
static_df = spark.read.parquet(f"abfss://source@storageaccountproject1.dfs.core.windows.net/{table_name}")
inferred_schema = static_df.schema

In [0]:
df = spark.readStream\
    .format("parquet")\
    .schema(inferred_schema)\
    .option("mergeSchema", "true")\
    .load(f"abfss://source@storageaccountproject1.dfs.core.windows.net/{table_name}")
            

In [0]:
df.writeStream\
  .format("delta")\
  .outputMode("append")\
  .option("checkpointLocation",f"abfss://source@storageaccountproject1.dfs.core.windows.net/checkpoints/{catalog_name}/{table_name}")\
  .trigger(once=True)\
  .toTable(f"{catalog_name}.bronze.{table_name}")